In [37]:
import requests
import pandas as pd
from tqdm import tqdm

In [38]:
zips_all = pd.read_json(
    "../../_reference/data/zips_reference_pop_gen.json"
).sort_values("population", ascending=False)

#### Get the most populous ZIPs and a sample of the rest

In [39]:
zips_top = zips_all.head(750)
zips_sample = zips_all.tail(len(zips_all) - 750).sample(750)
zips_df = pd.concat([zips_sample, zips_top]).reset_index(drop=True)

In [40]:
zips = zips_df['zip'].to_list()

In [41]:
headers = {
    'Accept': '*/*',
    'Accept-Language': 'en-US,en;q=0.9,es;q=0.8',
    'Connection': 'keep-alive',
    'Referer': 'https://stores.barnesandnoble.com/',
    'Sec-Fetch-Dest': 'empty',
    'Sec-Fetch-Mode': 'cors',
    'Sec-Fetch-Site': 'same-origin',
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36'
}

In [42]:
store_list = []

for zip_code in tqdm(zips):
    
    params = {
        'searchText': zip_code,
    }
    
    response = requests.get(
        'https://stores.barnesandnoble.com/_next/data/q_Dn38wVXgIqr66gMVyVY/index.json',
        params=params,
        headers=headers,
    )

    stores_json = response.json()['pageProps']['stores']['content']

    for store in stores_json:
        store_list.append(
            {
                'store_id': store['storeId'],
                'name': store['name'],
                'address': store['address1'],
                'address2': store.get('address2', None),
                'city': store['city'],
                'state': store['state'],
                'zip': store['zip'],
                'phone': store['phone'],
                'longitude': store['location'][0],
                'latitude': store['location'][1]
            }
        )

  0%|▎                                                                                                    | 4/1500 [00:02<17:54,  1.39it/s]


KeyError: 'content'

In [26]:
df = pd.DataFrame(store_list).drop_duplicates()
len(df)

96